# 1 — Data collection

**ADE-Sentinel** reads sentences from medical case reports and answers two questions:

1. **Stage 1 — does this sentence report an adverse drug event?** (a drug causing a bad
   side effect). Yes or no, for the whole sentence.
2. **Stage 2 — if yes, which words are the drug and which are the effect?**

To build that, the project needs two completely different kinds of data.

| | What it is | Labelled? | Used for |
|---|---|---|---|
| **ADE Corpus v2** | 20,896 sentences from case reports, each marked ADE / not-ADE, and 4,271 of them with the exact drug and effect character positions | yes | training and scoring both stages |
| **PubMed abstracts** | 159,975 raw medical abstracts, no labels at all | no | learning what medical words *mean* (notebook 3) |

This notebook covers where both came from.

In [1]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 90)
print("project root:", ROOT)

project root: E:\CSE\NLP Project


---

## 1.1 The labelled data: ADE Corpus v2

A public dataset on the Hugging Face Hub (`ade-benchmark-corpus/ade_corpus_v2`). It ships
in two configurations, and the project uses both:

- `Ade_corpus_v2_classification` — 23,516 rows, each a sentence plus a 0/1 label.
- `Ade_corpus_v2_drug_ade_relation` — 6,821 rows, each **one drug-effect pair** with
  character positions, so a sentence with three pairs appears three times.

Downloading it is one line:

```python
from datasets import load_dataset
clf = load_dataset("ade-benchmark-corpus/ade_corpus_v2", "Ade_corpus_v2_classification")
rel = load_dataset("ade-benchmark-corpus/ade_corpus_v2", "Ade_corpus_v2_drug_ade_relation")
```

### Two things that had to be measured, not assumed

**Duplicates.** 23,516 rows are only 20,896 distinct sentences. The duplication is *not*
spread evenly across the labels — it is concentrated in the positives — so removing it
changes the class balance materially:

| | ADE share |
|---|---|
| raw rows | 29.01% |
| after removing duplicate sentences | **20.44%** |

That second number is the one the models actually face, so it is the one the design has to
account for. No sentence carries conflicting labels across its duplicates, so dropping them
is safe.

**Relation rows are not duplicates.** In the relation config, the 6,821 rows are 4,271
distinct sentences — but the repeats are *different drug-effect pairs in the same
sentence*. A naive `drop_duplicates()` here would throw away 2,550 pairs, 37% of all the
Stage 2 supervision. They are grouped by sentence and their spans unioned instead.

### What the frozen splits look like

The train/dev/test split was made once, with seed 42, and saved to `data/splits/`. Those
files are committed to git, so every experiment in the project scored against the exact
same test set. Nothing re-splits at runtime.

In [2]:
splits = {name: pd.read_parquet(ROOT / "data" / "splits" / f"{name}.parquet")
          for name in ["stage1_train", "stage1_dev", "stage1_test",
                       "stage2_train", "stage2_dev", "stage2_test"]}

summary = pd.DataFrame([
    {"split": name,
     "rows": len(df),
     "columns": ", ".join(df.columns),
     "% ADE": f"{df['label'].mean():.2%}" if "label" in df else "-",
     "spans": int(df["n_spans"].sum()) if "n_spans" in df else "-"}
    for name, df in splits.items()])
summary

,split,rows,columns,% ADE,spans
0,stage1_train,14628,"text, label",20.44%,-
1,stage1_dev,3135,"text, label",20.45%,-
2,stage1_test,3133,"text, label",20.43%,-
3,stage2_train,2990,"text, spans, n_spans",-,7739
4,stage2_dev,641,"text, spans, n_spans",-,1639
5,stage2_test,640,"text, spans, n_spans",-,1636


Note the positive rate: 20.4% in every Stage 1 split, matching the corpus. The split was
*stratified* on the label so this could not drift.

And note that the Stage 2 splits are much smaller. Every Stage 2 sentence is a Stage 1
positive — Stage 2 only ever sees sentences that really do report an ADE.

In [3]:
print("--- a Stage 1 row (sentence + label) ---")
print(splits["stage1_train"].iloc[3].to_dict())

print("\n--- a Stage 2 row (sentence + character spans) ---")
row = splits["stage2_train"].loc[136]
print("text: ", row["text"])
print("spans:", row["spans"])
for start, end, label in json.loads(row["spans"]):
    print(f"   [{start}:{end}] {label:6s} -> {row['text'][start:end]!r}")

--- a Stage 1 row (sentence + label) ---
{'text': '"Retinoic acid syndrome" was prevented with short-time treatment of high dose (4 x 1.5 g/m2) cytarabine.', 'label': 0}

--- a Stage 2 row (sentence + character spans) ---
text:  A case of recall pneumonitis induced by gemcitabine is reported.
spans: [[10, 28, "EFFECT"], [40, 51, "DRUG"]]
   [10:28] EFFECT -> 'recall pneumonitis'
   [40:51] DRUG   -> 'gemcitabine'


That last block is the whole Stage 2 problem in miniature: the corpus gives *character
positions*, and a tagger needs one label per word. Notebook 2 does that conversion.

### Why one split is shared by both stages

Every relation sentence is also a classification row. If the two stages were split
independently, sentences used to *train* Stage 1 would end up in the *test* set of
Stage 2 — and the end-to-end measurement in notebook 6 would be meaningless. So the split
is computed once over the union of both configs and then projected onto each one.

---

## 1.2 The unlabelled data: 159,975 PubMed abstracts

The project's central claim is that **word embeddings trained on medical text beat
general-purpose English embeddings**. To test that, it has to train its own embeddings,
and for that it needs a large medical corpus. The ADE corpus (21k sentences) is nowhere
near big enough.

So abstracts were downloaded from PubMed through the NCBI E-utilities API.

### The search query

```python
QUERY = ('("drug-related side effects and adverse reactions"[MeSH]'
         ' OR "adverse effects"[Subheading]) AND hasabstract')
MIN_YEAR, MAX_YEAR = 2010, 2025
```

The MeSH term on its own returns 92,864 abstracts across all time — under the 100k target
— so the `adverse effects` subheading is unioned in. `hasabstract` filters server-side, so
no download is spent on a record with nothing to read.

### The one real obstacle: PubMed will not give you more than ~10,000 results

`esearch` hard-caps `retstart` at 9,998. Ask for a 30-year range and you get the first
10k and nothing else. The fix is to **search one year at a time**, splitting further
within a year when a window still returns too many:

```python
for year in range(MIN_YEAR, MAX_YEAR + 1):
    for mindate, maxdate in windows_for_year(year):   # halve until under the cap
        pmids = get_pmids(mindate, maxdate)
        for batch in chunks(pmids, 200):              # efetch takes 200 at a time
            yield from parse_articles(efetch(batch))
```

Each year is also capped at 10,000 records, so the corpus stays temporally balanced
instead of being all 2010-2012. 16 years x 10k gives the ~160k that arrived.

Each downloaded year is written to its own shard file, so an interrupted download resumes
instead of starting over. The shards are merged into `data/pubmed_corpus.jsonl`.

In [4]:
corpus_path = ROOT / "data" / "pubmed_corpus.jsonl"
print(f"{corpus_path.name}: {corpus_path.stat().st_size / 1e6:.0f} MB")

with corpus_path.open(encoding="utf-8") as fh:
    record = json.loads(fh.readline())

print("fields:", list(record))
print("pmid: ", record["pmid"])
print("title:", record["title"])
print("\nabstract:")
print(textwrap.fill(record["abstract"][:700], 95))

pubmed_corpus.jsonl: 285 MB
fields: ['pmid', 'title', 'abstract']
pmid:  20079996
title: Trauma and substance abuse: deadly consequences of intravenous percocet tablets.

abstract:
BACKGROUND: The prevalence of drug or alcohol addiction among trauma patients approaches 40%,
yet many require narcotics during admission for adequate pain control. Provider awareness is
the most reasonable option to avoid the devastating consequence of narcotic tablet injection.
OBJECTIVE: To illustrate the misuse of oral narcotics and to heighten provider awareness of a
potential cause for acute respiratory failure in recently discharged patients. CASE REPORT: A
20-year-old man was admitted to the hospital after an assault to the head and face. He was
discharged from the hospital with 30 oral Percocet® (Endo Pharmaceuticals, Newark, DE) tablets
after 24 h of observation. The day after disc


That is one of 159,975. No labels, no annotations — just medical English. Notebook 3 turns
these into word vectors.

---

## What this notebook produced

| Artefact | Contents |
|---|---|
| `data/splits/stage1_*.parquet` | 14,628 / 3,135 / 3,133 labelled sentences, frozen |
| `data/splits/stage2_*.parquet` | 2,990 / 641 / 640 sentences with character spans |
| `data/pubmed_corpus.jsonl` | 159,975 raw abstracts, 272 MB |

**Next:** [2 — Preprocessing](02_preprocessing.ipynb), where the text gets split into words
and the character spans become per-word tags.